[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdgordillob/aca_aci_collab/blob/main/notebooks/aci_lib_quickstart.ipynb)

# ACI-CO processed-data quickstart

Loads already-computed outputs of the Colombian Actuarial Climate Index (ACI-CO) pipeline -- baseline percentiles, per-region anomaly series, ENSO/SST indices, UNGRD validation tables, etc. -- through the `aci_lib` package in this repo.

This repo ships **code only** (`src/aci_lib/`, plus `ARCHITECTURE.tex`/`.pdf` describing the full ACI-CO pipeline this data comes from). The processed data itself is not committed here -- it's ~13GB across ~8,400 files, far too large for git -- so you supply it yourself below (Drive mount or upload).

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
IN_COLAB

## 1. Get the code

In Colab this clones the repo and puts `src/` on `sys.path`. Running this notebook from a local checkout instead, it just adds the sibling `../src`.

In [ ]:
import sys, os

if IN_COLAB:
    !git clone --depth 1 https://github.com/mdgordillob/aca_aci_collab.git
    sys.path.insert(0, "/content/aca_aci_collab/src")
else:
    sys.path.insert(0, os.path.abspath("../src"))

import aci_lib

## 2. Supply the data

`data/processed/` is not in this repo (see above). Pick **one**:

- **(a) Mount Google Drive**, if you've uploaded a copy of `data/processed/` there.
- **(b) Upload + unzip** a local copy directly into the Colab VM.
- **(c) Local checkout**: if you're running this notebook from inside a checkout of the source project (which has `data/processed/` on disk, just gitignored), skip both -- `aci_lib` finds it automatically.

In [ ]:
# (a) Drive mount -- edit DATA_DIR to wherever you put data/processed/ in your Drive.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/aca_data/processed"  # <-- edit this path
    aci_lib.set_data_dir(DATA_DIR)

In [ ]:
# (b) Upload + unzip instead of Drive -- uncomment to use.
# from google.colab import files
# uploaded = files.upload()  # pick a .zip of data/processed/
# zip_name = next(iter(uploaded))
# !unzip -q "{zip_name}" -d /content/aci_data
# aci_lib.set_data_dir("/content/aci_data/processed")

## 3. Browse the catalog

`list_datasets()` groups the raw files into loadable datasets -- e.g. the ~768 monthly NetCDF files under `anomalias_colombia/anomalies_temperature_*.nc` show up as a single `anomalias_colombia/anomalies_temperature` series.

In [ ]:
catalog = aci_lib.list_datasets()
print(len(catalog), "datasets")
catalog.head(20)

## 4. Load examples

`load(name)` returns a `pandas.DataFrame` for CSV/XLSX datasets and a lazy `xarray.Dataset` for single- or multi-file NetCDF datasets.

In [ ]:
# Monthly national temperature-anomaly series (tabular).
national_temp = aci_lib.load("anomalias_colombia/anomalies_temperature_combined")
national_temp.head()

In [ ]:
# The gridded fields underlying that series (lazy xarray.Dataset, ~768 files).
grid = aci_lib.load("anomalias_colombia/anomalies_temperature")
grid

In [ ]:
import matplotlib.pyplot as plt

grid["t_90"].isel(file=0).plot()
plt.title("T90 anomaly field -- first month in the series")
plt.show()

## 5. Reference

- `aci_lib.describe(name)` -- underlying file list and kind for one dataset.
- `ARCHITECTURE.pdf` (this repo) -- full pipeline description; Section 8 covers `aci_lib` itself, Section 4 the data-flow stages each dataset comes from.